In [96]:
#import SparkSession
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName('random_forest').getOrCreate()

In [97]:
#read the dataset
df=spark.read.csv('affairs.csv',inferSchema=True,header=True)

In [98]:
#check the shape of the data 
print((df.count(),len(df.columns)))

(6366, 6)


In [99]:
#printSchema
df.printSchema()

root
 |-- rate_marriage: integer (nullable = true)
 |-- age: double (nullable = true)
 |-- yrs_married: double (nullable = true)
 |-- children: double (nullable = true)
 |-- religious: integer (nullable = true)
 |-- affairs: integer (nullable = true)



In [100]:
#view the dataset
df.show(5)

+-------------+----+-----------+--------+---------+-------+
|rate_marriage| age|yrs_married|children|religious|affairs|
+-------------+----+-----------+--------+---------+-------+
|            5|32.0|        6.0|     1.0|        3|      0|
|            4|22.0|        2.5|     0.0|        2|      0|
|            3|32.0|        9.0|     3.0|        3|      1|
|            3|27.0|       13.0|     3.0|        1|      1|
|            4|22.0|        2.5|     0.0|        1|      1|
+-------------+----+-----------+--------+---------+-------+
only showing top 5 rows


In [101]:
#Exploratory Data Analysis
df.describe().select('summary','rate_marriage','age','yrs_married','children','religious').show()

+-------+------------------+------------------+-----------------+------------------+------------------+
|summary|     rate_marriage|               age|      yrs_married|          children|         religious|
+-------+------------------+------------------+-----------------+------------------+------------------+
|  count|              6366|              6366|             6366|              6366|              6366|
|   mean| 4.109644989004084|29.082862079798932| 9.00942507068803|1.3968740182218033|2.4261702796104303|
| stddev|0.9614295945655025| 6.847881883668817|7.280119972766412| 1.433470828560344|0.8783688402641785|
|    min|                 1|              17.5|              0.5|               0.0|                 1|
|    max|                 5|              42.0|             23.0|               5.5|                 4|
+-------+------------------+------------------+-----------------+------------------+------------------+



In [102]:
df.groupBy('affairs').count().show()

+-------+-----+
|affairs|count|
+-------+-----+
|      1| 2053|
|      0| 4313|
+-------+-----+



In [103]:
df.groupBy('rate_marriage').count().show()

+-------------+-----+
|rate_marriage|count|
+-------------+-----+
|            1|   99|
|            3|  993|
|            5| 2684|
|            4| 2242|
|            2|  348|
+-------------+-----+



In [104]:
df.groupBy('rate_marriage','affairs').count().orderBy('rate_marriage','affairs','count',ascending=True).show()

+-------------+-------+-----+
|rate_marriage|affairs|count|
+-------------+-------+-----+
|            1|      0|   25|
|            1|      1|   74|
|            2|      0|  127|
|            2|      1|  221|
|            3|      0|  446|
|            3|      1|  547|
|            4|      0| 1518|
|            4|      1|  724|
|            5|      0| 2197|
|            5|      1|  487|
+-------------+-------+-----+



In [105]:
df.groupBy('religious','affairs').count().orderBy('religious','affairs','count',ascending=True).show()

+---------+-------+-----+
|religious|affairs|count|
+---------+-------+-----+
|        1|      0|  613|
|        1|      1|  408|
|        2|      0| 1448|
|        2|      1|  819|
|        3|      0| 1715|
|        3|      1|  707|
|        4|      0|  537|
|        4|      1|  119|
+---------+-------+-----+



In [106]:
df.groupBy('children','affairs').count().orderBy('children','affairs','count',ascending=True).show()

+--------+-------+-----+
|children|affairs|count|
+--------+-------+-----+
|     0.0|      0| 1912|
|     0.0|      1|  502|
|     1.0|      0|  747|
|     1.0|      1|  412|
|     2.0|      0|  873|
|     2.0|      1|  608|
|     3.0|      0|  460|
|     3.0|      1|  321|
|     4.0|      0|  197|
|     4.0|      1|  131|
|     5.5|      0|  124|
|     5.5|      1|   79|
+--------+-------+-----+



In [107]:
df.groupBy('affairs').mean().show()

+-------+------------------+------------------+------------------+------------------+------------------+------------+
|affairs|avg(rate_marriage)|          avg(age)|  avg(yrs_married)|     avg(children)|    avg(religious)|avg(affairs)|
+-------+------------------+------------------+------------------+------------------+------------------+------------+
|      1|3.6473453482708234|30.537018996590355|11.152459814905017|1.7289332683877252| 2.261568436434486|         1.0|
|      0| 4.329700904242986| 28.39067934152562| 7.989334569904939|1.2388128912589844|2.5045212149316023|         0.0|
+-------+------------------+------------------+------------------+------------------+------------------+------------+



In [108]:
from pyspark.ml.feature import VectorAssembler

In [109]:
df_assembler = VectorAssembler(inputCols=['rate_marriage', 'age', 'yrs_married', 'children', 'religious'], outputCol="features")
df = df_assembler.transform(df)

In [110]:
df.printSchema()

root
 |-- rate_marriage: integer (nullable = true)
 |-- age: double (nullable = true)
 |-- yrs_married: double (nullable = true)
 |-- children: double (nullable = true)
 |-- religious: integer (nullable = true)
 |-- affairs: integer (nullable = true)
 |-- features: vector (nullable = true)



In [111]:
df.select(['features','affairs']).show(10,False)

+-----------------------+-------+
|features               |affairs|
+-----------------------+-------+
|[5.0,32.0,6.0,1.0,3.0] |0      |
|[4.0,22.0,2.5,0.0,2.0] |0      |
|[3.0,32.0,9.0,3.0,3.0] |1      |
|[3.0,27.0,13.0,3.0,1.0]|1      |
|[4.0,22.0,2.5,0.0,1.0] |1      |
|[4.0,37.0,16.5,4.0,3.0]|1      |
|[5.0,27.0,9.0,1.0,1.0] |1      |
|[4.0,27.0,9.0,0.0,2.0] |1      |
|[5.0,37.0,23.0,5.5,2.0]|1      |
|[5.0,37.0,23.0,5.5,2.0]|1      |
+-----------------------+-------+
only showing top 10 rows


In [112]:
#select data for building model
model_df=df.select(['features','affairs'])

In [113]:
train_df,test_df=model_df.randomSplit([0.75,0.25])

In [114]:
train_df.count()

4758

In [115]:
train_df.groupBy('affairs').count().show()

+-------+-----+
|affairs|count|
+-------+-----+
|      1| 1540|
|      0| 3218|
+-------+-----+



In [116]:
test_df.groupBy('affairs').count().show()

+-------+-----+
|affairs|count|
+-------+-----+
|      1|  513|
|      0| 1095|
+-------+-----+



In [117]:
from pyspark.ml.classification import RandomForestClassifier

In [118]:
rf_classifier=RandomForestClassifier(labelCol='affairs',numTrees=50).fit(train_df)
#.fit(train_df)   This trains (fits) the Random Forest on your training dataset:

In [ ]:
'''
results=rf_classifier.evaluate(test_df).predictions
results.select(['affairs','prediction']).show(10,False)
true_postives = results[(results.affairs == 1) & (results.prediction == 1)].count()
true_negatives = results[(results.affairs == 0) & (results.prediction == 0)].count()
false_positives = results[(results.affairs == 0) & (results.prediction == 1)].count()
false_negatives = results[(results.affairs == 1) & (results.prediction == 0)].count()

print (true_postives)
print (true_negatives)
print (false_positives)
print (false_negatives)
print(true_postives+true_negatives+false_positives+false_negatives)
print (results.count())

recall = float(true_postives)/(true_postives + false_negatives)
print(recall)

precision = float(true_postives) / (true_postives + false_positives)
print(precision)


accuracy=float((true_postives+true_negatives) /(results.count()))
print(accuracy)
'''

+-------+----------+
|affairs|prediction|
+-------+----------+
|1      |1.0       |
|1      |1.0       |
|0      |1.0       |
|0      |1.0       |
|0      |1.0       |
|1      |1.0       |
|0      |1.0       |
|0      |1.0       |
|1      |1.0       |
|1      |1.0       |
+-------+----------+
only showing top 10 rows
175
977
118
338
1608
1608
0.341130604288499
0.5972696245733788
0.7164179104477612


In [ ]:
rf_predictions=rf_classifier.transform(test_df)


'''  
This applies the trained Random Forest model to the test dataset.
transform(test_df) adds new prediction columns to test_df, such as:
    -rawPrediction
    -probability
    -prediction ← the final predicted class

PySpark wraps decision-tree-based models in a probabilistic classifier API, which automatically produces these columns.

A single Decision Tree normally gives only a class label.
But in PySpark:
	DecisionTreeClassifier
	RandomForestClassifier	
	GBTClassifier
all follow the same ML API, which produces:

-rawPrediction
    For Random Forest:
    rawPrediction = counts of votes from all trees


Example (binary classification):
50 trees predict:
Class 0 → 18 votes
Class 1 → 32 votes
rawPrediction will be:
[18.0, 32.0]

-probability
    This converts the vote counts into probabilities:
        probability = rawPrediction / sum(rawPrediction)
Using the same example:
probability = [18/50, 32/50] = [0.36, 0.64]
So the model says:
36% chance class 0
64% chance class 1
-prediction
This is simply:
prediction = argmax(probability)   // “the index of the largest probability value.”
                                    In classification, this chooses the class with the highest predicted probability.
So prediction = 1 (because 0.64 is higher).
'''

'  \nThis applies the trained Random Forest model to the test dataset.\ntransform(test_df) adds new prediction columns to test_df, such as:\n    -rawPrediction\n    -probability\n    -prediction ← the final predicted class\n\nPySpark wraps decision-tree-based models in a probabilistic classifier API, which automatically produces these columns.\n\nA single Decision Tree normally gives only a class label.\nBut in PySpark:\n\tDecisionTreeClassifier\n\tRandomForestClassifier\t\n\tGBTClassifier\nall follow the same ML API, which produces:\n\n-rawPrediction\nFor Random Forest:\nrawPrediction = counts of votes from all trees\n\n\nExample (binary classification):\n50 trees predict:\nClass 0 → 18 votes\nClass 1 → 32 votes\nrawPrediction will be:\n[18.0, 32.0]\n\n-probability\nThis converts the vote counts into probabilities:\nprobability = rawPrediction / sum(rawPrediction)\nUsing the same example:\nprobability = [18/50, 32/50] = [0.36, 0.64]\nSo the model says:\n36% chance class 0\n64% chance 

In [121]:
rf_predictions.show()

+--------------------+-------+--------------------+--------------------+----------+
|            features|affairs|       rawPrediction|         probability|prediction|
+--------------------+-------+--------------------+--------------------+----------+
|[1.0,22.0,2.5,1.0...|      1|[17.3705801837822...|[0.34741160367564...|       1.0|
|[1.0,22.0,2.5,1.0...|      1|[17.3705801837822...|[0.34741160367564...|       1.0|
|[1.0,22.0,2.5,1.0...|      0|[21.2764210635048...|[0.42552842127009...|       1.0|
|[1.0,27.0,6.0,1.0...|      0|[15.6059990617183...|[0.31211998123436...|       1.0|
|[1.0,27.0,6.0,3.0...|      0|[13.5521658112407...|[0.27104331622481...|       1.0|
|[1.0,27.0,9.0,1.0...|      1|[15.2454706217525...|[0.30490941243505...|       1.0|
|[1.0,27.0,9.0,4.0...|      0|[13.8421482048160...|[0.27684296409632...|       1.0|
|[1.0,32.0,2.5,1.0...|      0|[21.2331494096775...|[0.42466298819355...|       1.0|
|[1.0,32.0,13.0,2....|      1|[13.7260914096420...|[0.27452182819284...|    

In [122]:
rf_predictions.groupBy('prediction').count().show()

+----------+-----+
|prediction|count|
+----------+-----+
|       0.0| 1315|
|       1.0|  293|
+----------+-----+



In [123]:
rf_predictions.select(['probability','affairs','prediction']).show(10,False)

+----------------------------------------+-------+----------+
|probability                             |affairs|prediction|
+----------------------------------------+-------+----------+
|[0.34741160367564494,0.652588396324355] |1      |1.0       |
|[0.34741160367564494,0.652588396324355] |1      |1.0       |
|[0.42552842127009705,0.574471578729903] |0      |1.0       |
|[0.3121199812343678,0.6878800187656321] |0      |1.0       |
|[0.27104331622481465,0.7289566837751854]|0      |1.0       |
|[0.30490941243505026,0.6950905875649498]|1      |1.0       |
|[0.27684296409632075,0.7231570359036793]|0      |1.0       |
|[0.424662988193552,0.575337011806448]   |0      |1.0       |
|[0.27452182819284005,0.7254781718071599]|1      |1.0       |
|[0.27452182819284005,0.7254781718071599]|1      |1.0       |
+----------------------------------------+-------+----------+
only showing top 10 rows


In [124]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

In [125]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
'''MulticlassClassificationEvaluator is used to measure how good a classification model is when the problem has more than two classes.
It calculates evaluation metrics for classification models such as:
accuracy
precision
recall
F1-score
weighted metrics (weighted by class frequency)

This evaluator is used when your label column has 3 or more possible values, for example:

0, 1, 2

Low, Medium, High

1, 2, 3, 4, 5

as in the dataset (affairs) is multiclass because affairs can take values 0, 1, 2, 3...'''

'MulticlassClassificationEvaluator is used to measure how good a classification model is when the problem has more than two classes.\nIt calculates evaluation metrics for classification models such as:\naccuracy\nprecision\nrecall\nF1-score\nweighted metrics (weighted by class frequency)\n\nThis evaluator is used when your label column has 3 or more possible values, for example:\n\n0, 1, 2\n\nLow, Medium, High\n\n1, 2, 3, 4, 5\n\nas in the dataset (affairs) is multiclass because affairs can take values 0, 1, 2, 3...'

In [126]:
rf_accuracy=MulticlassClassificationEvaluator(labelCol='affairs',metricName='accuracy').evaluate(rf_predictions)

In [127]:
print('The accuracy of RF on test data is {0:.0%}'.format(rf_accuracy))

The accuracy of RF on test data is 72%


In [128]:
print(rf_accuracy)

0.7164179104477612


In [129]:
rf_precision=MulticlassClassificationEvaluator(labelCol='affairs',metricName='weightedPrecision').evaluate(rf_predictions)

In [130]:
print('The precision rate on test data is {0:.0%}'.format(rf_precision))

The precision rate on test data is 70%


In [131]:
rf_precision

0.6964843569174463

In [132]:
rf_auc=BinaryClassificationEvaluator(labelCol='affairs').evaluate(rf_predictions)
'''
By default, Spark uses: metricName = 'areaUnderROC'
AUC — Area Under the Receiver Operating Characteristic Curve
AUC measures:
How well the model separates Class 0 vs Class 1

1.0 → perfect classifier
0.5 → random guessing
< 0.5 → model is predicting opposite (bad)
'''

"\nBy default, Spark uses: metricName = 'areaUnderROC'\nAUC — Area Under the Receiver Operating Characteristic Curve\nAUC measures:\nHow well the model separates Class 0 vs Class 1\n\n1.0 → perfect classifier\n0.5 → random guessing\n< 0.5 → model is predicting opposite (bad)\n"

In [133]:
print(rf_auc)

0.7406009951311558


In [134]:
# Feature importance

In [135]:
rf_classifier.featureImportances

SparseVector(5, {0: 0.6104, 1: 0.0248, 2: 0.2387, 3: 0.0617, 4: 0.0644})

In [136]:
df.schema["features"].metadata["ml_attr"]["attrs"]

{'numeric': [{'name': 'rate_marriage', 'idx': 0},
  {'name': 'age', 'idx': 1},
  {'name': 'yrs_married', 'idx': 2},
  {'name': 'children', 'idx': 3},
  {'name': 'religious', 'idx': 4}]}

In [137]:
# Save the model 

In [ ]:
rf_classifier.write().overwrite().save("RF_model")

In [139]:
from pyspark.ml.classification import RandomForestClassificationModel

In [140]:
rf=RandomForestClassificationModel.load("RF_model")

In [141]:
model_preditions=rf.transform(test_df)

In [142]:
model_preditions.show()

+--------------------+-------+--------------------+--------------------+----------+
|            features|affairs|       rawPrediction|         probability|prediction|
+--------------------+-------+--------------------+--------------------+----------+
|[1.0,22.0,2.5,1.0...|      1|[17.3705801837822...|[0.34741160367564...|       1.0|
|[1.0,22.0,2.5,1.0...|      1|[17.3705801837822...|[0.34741160367564...|       1.0|
|[1.0,22.0,2.5,1.0...|      0|[21.2764210635048...|[0.42552842127009...|       1.0|
|[1.0,27.0,6.0,1.0...|      0|[15.6059990617183...|[0.31211998123436...|       1.0|
|[1.0,27.0,6.0,3.0...|      0|[13.5521658112407...|[0.27104331622481...|       1.0|
|[1.0,27.0,9.0,1.0...|      1|[15.2454706217525...|[0.30490941243505...|       1.0|
|[1.0,27.0,9.0,4.0...|      0|[13.8421482048160...|[0.27684296409632...|       1.0|
|[1.0,32.0,2.5,1.0...|      0|[21.2331494096775...|[0.42466298819355...|       1.0|
|[1.0,32.0,13.0,2....|      1|[13.7260914096420...|[0.27452182819284...|    